# Machine Learning 2026 — Assignment 4
**Roll number:** 2024514  
**Reproducibility:** all randomness derives from `ROLL_NO` and `SEED`.  
**Important:** CIFAR-10 is loaded using the assignment-prescribed torchvision dataset. The runtime used to prepare this file had no network access, so CIFAR-dependent cells must be run where CIFAR-10 is available. Do not submit fabricated outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
ROLL_NO = 2024514
SEED = ROLL_NO
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)

## P1. Activation functions and derivatives

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh(z):
    return np.tanh(np.asarray(z, dtype=np.float64))

def tanh_prime(z):
    t = tanh(z)
    return 1.0 - t*t

def relu(z):
    return np.maximum(0.0, np.asarray(z, dtype=np.float64))

def relu_prime(z):
    z = np.asarray(z)
    return (z > 0).astype(np.float64)

def leaky_relu(z, alpha=0.1):
    z = np.asarray(z, dtype=np.float64)
    return np.where(z > 0, z, alpha*z)

def leaky_relu_prime(z, alpha=0.1):
    z = np.asarray(z)
    return np.where(z > 0, 1.0, alpha)

def softplus(z):
    z = np.asarray(z, dtype=np.float64)
    return np.logaddexp(0.0, z)

def softplus_prime(z):
    return sigmoid(z)

def num_deriv(f, z, eps=1e-6):
    return (f(z + eps) - f(z - eps)) / (2.0*eps)

zcheck = np.array([-3., -1.5, -.5, .5, 1.5, 3.])
pairs = [(sigmoid,sigmoid_prime),(tanh,tanh_prime),(relu,relu_prime),
         (leaky_relu,leaky_relu_prime),(softplus,softplus_prime)]
for f, fp in pairs:
    err = np.max(np.abs(num_deriv(f,zcheck)-fp(zcheck)))
    print(f.__name__, "max abs error:", err, "PASS" if err < 1e-6 else "FAIL")
    assert err < 1e-6
zg = np.linspace(-5,5,400)
fig, ax = plt.subplots(1,2,figsize=(13,4))
for f,fp in pairs:
    ax[0].plot(zg,f(zg),label=f.__name__)
    ax[1].plot(zg,fp(zg),label=f.__name__ + "'")
    print(f.__name__, "maximum derivative on grid:", np.max(fp(zg)))
ax[0].set_title("Activation functions"); ax[1].set_title("Derivatives")
for a in ax: a.grid(True); a.legend()
plt.tight_layout(); plt.show()
assert np.max(np.abs(tanh(zg)-(2*sigmoid(2*zg)-1))) < 1e-12
print("tanh identity: PASS")
for val in [-2.,0.,2.]:
    print("z =",val, "sigmoid' =",sigmoid_prime(val),"tanh' =",tanh_prime(val),
          "ReLU' =",relu_prime(val),"leaky ReLU' =",leaky_relu_prime(val),
          "softplus' =",softplus_prime(val))

## P2. Hand backpropagation (tanh hidden layer)

In [ ]:
x2 = np.array([1.,2.]); y2 = 1.
W12 = np.array([[.5,.25],[-.5,.1]])
b12 = np.array([.1,.2]); w22 = np.array([.6,-.4]); b22 = .1
z2 = W12 @ x2 + b12
h2 = np.tanh(z2)
u2 = w22 @ h2 + b22
yhat2 = sigmoid(np.array(u2)).item()
L2 = -np.log(yhat2)
du2 = yhat2-y2
dw22 = du2*h2
db22 = du2
dh2 = du2*w22
dz2 = dh2*(1-h2*h2)
dW12 = np.outer(dz2,x2)
db12 = dz2.copy()
print("z =",z2,"h =",h2,"u =",u2,"yhat =",yhat2,"loss =",L2)
print("dL/du =",du2,"dL/dw2 =",dw22,"dL/db2 =",db22)
print("dL/dh =",dh2,"dL/dz =",dz2,"dL/dW1 =\n",dW12,"dL/db1 =",db12)
eta2=.5
W12n=W12-eta2*dW12; b12n=b12-eta2*db12
w22n=w22-eta2*dw22; b22n=b22-eta2*db22
yn=sigmoid(np.array(w22n @ np.tanh(W12n@x2+b12n)+b22n)).item()
Ln=-np.log(yn)
print("updated loss =",Ln, "decreased:", Ln<L2)
assert Ln < L2

**P2 handwritten requirement:** The derivation and numerical values are included in the accompanying report as a clearly separated hand-solution-style section. Check whether your instructor requires literal pen-and-paper handwriting; if so, transcribe this section by hand before submission.

## P3. Stable softmax and cross-entropy

In [ ]:
def softmax(Z):
    Z=np.asarray(Z,dtype=np.float64)
    shifted=Z-np.max(Z,axis=1,keepdims=True)
    E=np.exp(shifted)
    return E/np.sum(E,axis=1,keepdims=True)

def cross_entropy(P,y):
    P=np.asarray(P,dtype=np.float64); y=np.asarray(y,dtype=int)
    return float(-np.mean(np.log(P[np.arange(len(y)),y]+1e-12)))

def softmax_ce_grad(Z,y):
    Z=np.asarray(Z,dtype=np.float64); y=np.asarray(y,dtype=int)
    P=softmax(Z); G=P.copy()
    G[np.arange(len(y)),y]-=1
    return G/len(y)

Z0=np.array([[2.,1.,.1]]); y0=np.array([1])
P0=softmax(Z0)
print("checkpoint probabilities:",P0,"loss:",cross_entropy(P0,y0))
assert abs(P0.sum(axis=1)[0]-1)<1e-12
rg=np.random.default_rng(SEED)
Zr=rg.normal(size=(5,4)); yr=rg.integers(0,4,size=5)
G=softmax_ce_grad(Zr,yr); eps=1e-6
Gnum=np.zeros_like(Zr)
for i in range(5):
    for j in range(4):
        Zp=Zr.copy(); Zm=Zr.copy(); Zp[i,j]+=eps; Zm[i,j]-=eps
        Gnum[i,j]=(cross_entropy(softmax(Zp),yr)-cross_entropy(softmax(Zm),yr))/(2*eps)
err=np.max(np.abs(G-Gnum))
print("gradient max abs error:",err,"PASS" if err<1e-6 else "FAIL")
assert err<1e-6
print("row gradient sums:",G.sum(axis=1))
print("shift invariance:",np.max(np.abs(softmax(Zr+123.4)-softmax(Zr))))
with np.errstate(over='ignore',invalid='ignore'):
    naive=np.exp(np.array([[1000.,1001.,1002.]]))
    naive=naive/naive.sum(axis=1,keepdims=True)
print("naive softmax:",naive,"stable:",softmax(np.array([[1000.,1001.,1002.]])))

## P4. Data pipelines — Wine and CIFAR-10

In [ ]:
from sklearn.datasets import load_wine
wine=load_wine()
Xw=np.asarray(wine.data,dtype=np.float64); yw=np.asarray(wine.target,dtype=np.int64)
perm=np.random.default_rng(SEED).permutation(len(Xw))
ntrain=int(.8*len(Xw)); tr_idx=perm[:ntrain]; va_idx=perm[ntrain:]
X_train_raw=Xw[tr_idx]; X_val_raw=Xw[va_idx]
wine_mean=X_train_raw.mean(axis=0); wine_std=X_train_raw.std(axis=0)
wine_std[wine_std==0]=1.0
X_train=(X_train_raw-wine_mean)/wine_std
X_val=(X_val_raw-wine_mean)/wine_std
y_train=yw[tr_idx]; y_val=yw[va_idx]
print("Wine train/val shapes:",X_train.shape,X_val.shape)
print("Wine train means (standardised):",X_train.mean(axis=0))
print("Wine train stds (standardised):",X_train.std(axis=0))

In [ ]:
# CIFAR-10: exact assignment sampling protocol.
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
cifar_train=CIFAR10(root="./data",train=True,download=True)
cifar_test=CIFAR10(root="./data",train=False,download=True)
cifar_rng=np.random.default_rng(SEED)
chosen=cifar_rng.choice(len(cifar_train),size=10000,replace=False)
train_ids=chosen[:8000]; val_ids=chosen[8000:10000]
def cifar_arrays(dataset, ids):
    imgs=np.stack([np.asarray(dataset[i][0],dtype=np.float32)/255.0 for i in ids])
    labels=np.array([dataset[i][1] for i in ids],dtype=np.int64)
    return imgs,labels
# PIL images -> NHWC, then channel-first as required.
Xc_train_raw,yc_train=cifar_arrays(cifar_train,train_ids)
Xc_val_raw,yc_val=cifar_arrays(cifar_train,val_ids)
Xc_test_raw,yc_test=cifar_arrays(cifar_test,np.arange(2000))
channel_mean=Xc_train_raw.mean(axis=(0,1,2))
channel_std=Xc_train_raw.std(axis=(0,1,2))
channel_std[channel_std==0]=1
def norm_flat(a):
    a=(a-channel_mean[None,None,None,:])/channel_std[None,None,None,:]
    return a.transpose(0,3,1,2).reshape(len(a),3072).astype(np.float32)
Xc_train=norm_flat(Xc_train_raw); Xc_val=norm_flat(Xc_val_raw); Xc_test=norm_flat(Xc_test_raw)
print("CIFAR shapes:",Xc_train.shape,Xc_val.shape,Xc_test.shape)
print("channel mean/std before:",channel_mean,channel_std)
print("normalised training mean/std:",Xc_train.reshape(-1,3,32,32).mean((0,2,3)),
      Xc_train.reshape(-1,3,32,32).std((0,2,3)))
class_names=cifar_train.classes
fig,axs=plt.subplots(2,5,figsize=(12,5))
for c,ax in enumerate(axs.flat):
    ix=np.where(yc_train==c)[0][0]
    ax.imshow(Xc_train_raw[ix]); ax.set_title(class_names[c]); ax.axis("off")
plt.tight_layout(); plt.show()
print("training class counts:",np.bincount(yc_train,minlength=10))

## P5. NumPy MLP forward pass on Wine

In [ ]:
def init_mlp(sizes,seed,he=False):
    r=np.random.default_rng(seed); Ws=[]; bs=[]
    for fan_in,fan_out in zip(sizes[:-1],sizes[1:]):
        scale=np.sqrt(2.0/fan_in) if he else np.sqrt(2.0/fan_in)
        Ws.append(r.normal(0,scale,size=(fan_in,fan_out)))
        bs.append(np.zeros(fan_out))
    return Ws,bs
def relu_np(z): return np.maximum(z,0)
def forward(X,Ws,bs,activation="relu"):
    As=[X]; Zs=[]; A=X
    for l,(W,b) in enumerate(zip(Ws,bs)):
        Z=A@W+b; Zs.append(Z)
        if l<len(Ws)-1:
            A=relu_np(Z) if activation=="relu" else (np.tanh(Z) if activation=="tanh" else sigmoid(Z))
        else: A=Z
        As.append(A)
    return Zs,As
def softmax_np(z):
    e=np.exp(z-np.max(z,axis=1,keepdims=True)); return e/e.sum(axis=1,keepdims=True)
def loss_np(X,y,Ws,bs,activation="relu",lam=0.):
    Zs,As=forward(X,Ws,bs,activation)
    P=softmax_np(Zs[-1])
    ce=-np.log(P[np.arange(len(y)),y]+1e-12).mean()
    return float(ce+lam/2*sum((W*W).sum() for W in Ws)),Zs,As
W5,b5=init_mlp([13,16,8,3],SEED)
print("parameter formula/count:",sum(a*b+b for a,b in zip([13,16,8],[16,8,3])),
      sum(W.size+b.size for W,b in zip(W5,b5)))
Z5,A5=forward(X_train[:5],W5,b5)
for i,(z,a) in enumerate(zip(Z5,A5[1:]),1): print("layer",i,"Z",z.shape,"A",a.shape)
l5,_,_=loss_np(X_train,y_train,W5,b5)
print("initial loss vs ln(3):",l5,np.log(3))
W5small=[w.copy() for w in W5]; W5small[-1]*=.01
print("last-layer weights x .01 loss:",loss_np(X_train,y_train,W5small,b5)[0])
for act in ["relu","tanh","sigmoid"]:
    _,aa=forward(X_train,W5,b5,act)
    print(act,"first hidden mean/std:",aa[1].mean(),aa[1].std())
# Vectorisation check
batch=forward(X_train[:5],W5,b5)[0][-1]
single=np.vstack([forward(X_train[i:i+1],W5,b5)[0][-1] for i in range(5)])
diff=np.max(np.abs(batch-single)); print("vectorisation diff:",diff,"PASS" if diff<1e-12 else "FAIL")
assert diff<1e-12

## P6. NumPy backpropagation, gradient checks and overfit test

In [ ]:
def backward(X,y,Ws,bs,Zs,As,lam=0.,activation="relu"):
    n=len(y); P=softmax_np(Zs[-1]); Y=np.zeros_like(P); Y[np.arange(n),y]=1
    dZ=(P-Y)/n; dWs=[None]*len(Ws); dbs=[None]*len(Ws); dZs=[None]*len(Ws)
    for l in reversed(range(len(Ws))):
        dZs[l]=dZ.copy()
        dWs[l]=As[l].T@dZ+lam*Ws[l]
        dbs[l]=dZ.sum(axis=0)
        if l>0:
            dA=dZ@Ws[l].T
            if activation=="relu": deriv=(Zs[l-1]>0).astype(float)
            elif activation=="tanh": deriv=1-np.tanh(Zs[l-1])**2
            else:
                s=sigmoid(Zs[l-1]); deriv=s*(1-s)
            dZ=dA*deriv
    return dWs,dbs,dZs
def objective_params(X,y,Ws,bs,lam=0.):
    return loss_np(X,y,Ws,bs,lam=lam)[0]
W6,b6=init_mlp([13,5,4,3],SEED)
l6,Z6,A6=loss_np(X_train[:8],y_train[:8],W6,b6,lam=.01)
dW6,db6,dZ6=backward(X_train[:8],y_train[:8],W6,b6,Z6,A6,lam=.01)
# Check selected entries from every parameter array
eps=1e-6; relerrs=[]
for li in range(len(W6)):
    for arr,grad,isweight in [(W6[li],dW6[li],True),(b6[li],db6[li],False)]:
        for ix in list(np.ndindex(arr.shape))[:min(arr.size,8)]:
            old=arr[ix]; arr[ix]=old+eps; lp=objective_params(X_train[:8],y_train[:8],W6,b6,.01)
            arr[ix]=old-eps; lm=objective_params(X_train[:8],y_train[:8],W6,b6,.01)
            arr[ix]=old
            gn=(lp-lm)/(2*eps); g=grad[ix]
            relerrs.append(abs(g-gn)/max(1e-8,abs(g)+abs(gn)))
print("gradient-check worst relative error:",max(relerrs))
# Inject each requested bug independently and measure its worst relative gradient-check error.
def backward_with_bug(X,y,Ws,bs,Zs,As,lam=0.,bug=0):
    n=len(y); P=softmax_np(Zs[-1]); Y=np.zeros_like(P); Y[np.arange(n),y]=1
    dZ=P-Y if bug==1 else (P-Y)/n
    dWs=[None]*len(Ws); dbs=[None]*len(Ws)
    for l in reversed(range(len(Ws))):
        prev=As[l]
        if bug==2 and l>0:
            prev=Zs[l-1]
        dWs[l]=prev.T@dZ+lam*Ws[l]
        dbs[l]=dZ.sum(axis=0)
        if l>0:
            dA=dZ@Ws[l].T
            if bug==3:
                dZ=dA
            else:
                dZ=dA*(Zs[l-1]>0)
    return dWs,dbs

# Numerical reference gradients for every parameter in this small network.
def all_numeric_grads(X,y,Ws,bs,lam=0.,eps=1e-6):
    ngw=[]; ngb=[]
    for li in range(len(Ws)):
        gw=np.zeros_like(Ws[li]); gb=np.zeros_like(bs[li])
        for ix in np.ndindex(Ws[li].shape):
            old=Ws[li][ix]; Ws[li][ix]=old+eps
            lp=loss_np(X,y,Ws,bs,lam=lam)[0]
            Ws[li][ix]=old-eps; lm=loss_np(X,y,Ws,bs,lam=lam)[0]
            Ws[li][ix]=old; gw[ix]=(lp-lm)/(2*eps)
        for ix in np.ndindex(bs[li].shape):
            old=bs[li][ix]; bs[li][ix]=old+eps
            lp=loss_np(X,y,Ws,bs,lam=lam)[0]
            bs[li][ix]=old-eps; lm=loss_np(X,y,Ws,bs,lam=lam)[0]
            bs[li][ix]=old; gb[ix]=(lp-lm)/(2*eps)
        ngw.append(gw); ngb.append(gb)
    return ngw,ngb

numW,numb=all_numeric_grads(X_train[:8],y_train[:8],W6,b6,lam=.01)
for bug_id,description in [(1,"omit division by batch size"),(2,"use Z instead of A in dW"),(3,"omit activation derivative")]:
    bw,bbug=backward_with_bug(X_train[:8],y_train[:8],W6,b6,Z6,A6,lam=.01,bug=bug_id)
    errors=[]
    for g,gn in zip(bw,numW):
        errors.extend((np.abs(g-gn)/np.maximum(1e-8,np.abs(g)+np.abs(gn))).ravel())
    for g,gn in zip(bbug,numb):
        errors.extend((np.abs(g-gn)/np.maximum(1e-8,np.abs(g)+np.abs(gn))).ravel())
    print(description,"worst relative error:",max(errors))
# 8-sample plain-GD overfit experiment
Wo,bo=init_mlp([13,8,3],SEED)
steps=None
for step in range(501):
    lo,zo,ao=loss_np(X_train[:8],y_train[:8],Wo,bo)
    if lo<.01: steps=step; break
    gwo,gbo,_=backward(X_train[:8],y_train[:8],Wo,bo,zo,ao)
    Wo=[w-.5*g for w,g in zip(Wo,gwo)]; bo=[b-.5*g for b,g in zip(bo,gbo)]
print("overfit steps (<.01; None means not reached in 500):",steps)

## P7. NumPy MLP on CIFAR-10

In [ ]:
def init_cifar(seed,lam=0.):
    r=np.random.default_rng(seed); sizes=[3072,256,128,10]
    Ws=[r.normal(0,np.sqrt(2/sizes[i]),(sizes[i],sizes[i+1])).astype(np.float32) for i in range(3)]
    bs=[np.zeros(s,dtype=np.float32) for s in sizes[1:]]
    return Ws,bs
def cifar_eval(X,y,Ws,bs,batch=256):
    total=0.; correct=0
    for st in range(0,len(y),batch):
        xb=X[st:st+batch]; yb=y[st:st+batch]
        Z,A=forward(xb,Ws,bs); P=softmax_np(Z[-1])
        total+=-np.log(P[np.arange(len(yb)),yb]+1e-12).sum()
        correct+=(P.argmax(1)==yb).sum()
    return total/len(y),correct/len(y)
def train_cifar(Ws,bs,epochs=15,lr=.01,momentum=.9,lam=0.,seed=SEED):
    r=np.random.default_rng(seed); vW=[np.zeros_like(w) for w in Ws]; vb=[np.zeros_like(b) for b in bs]
    hist={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}; best=None; bestacc=-1
    for ep in range(epochs):
        order=r.permutation(len(yc_train))
        for st in range(0,len(order),128):
            ids=order[st:st+128]; xb=Xc_train[ids]; yb=yc_train[ids]
            _,zs,acts=loss_np(xb,yb,Ws,bs)
            gw,gb,_=backward(xb,yb,Ws,bs,zs,acts,lam)
            for j in range(len(Ws)):
                vW[j]=momentum*vW[j]+gw[j]; vb[j]=momentum*vb[j]+gb[j]
                Ws[j]-=lr*vW[j]; bs[j]-=lr*vb[j]
        tl,ta=cifar_eval(Xc_train,yc_train,Ws,bs); vl,va=cifar_eval(Xc_val,yc_val,Ws,bs)
        hist["train_loss"].append(tl); hist["train_acc"].append(ta)
        hist["val_loss"].append(vl); hist["val_acc"].append(va)
        if va>bestacc: bestacc=va; best=([w.copy() for w in Ws],[b.copy() for b in bs])
        print("epoch",ep+1,"train",tl,ta,"val",vl,va)
    return hist,best
W7,b7=init_cifar(SEED)
print("P7 parameter count:",sum(w.size+b.size for w,b in zip(W7,b7)))
print("initial loss:",cifar_eval(Xc_train,yc_train,W7,b7)[0],"ln(10):",np.log(10))
# 64-sample overfit test
Wo7,bo7=init_cifar(SEED); Wo7=[w.copy() for w in Wo7]; bo7=[b.copy() for b in bo7]
for step in range(501):
    ll,zz,aa=loss_np(Xc_train[:64],yc_train[:64],Wo7,bo7)
    if ll<.05: break
    gw,gb,_=backward(Xc_train[:64],yc_train[:64],Wo7,bo7,zz,aa)
    for j in range(3): Wo7[j]-=.05*gw[j]; bo7[j]-=.05*gb[j]
print("64-sample overfit steps:",step,"loss:",ll)
hist7,best7=train_cifar(W7,b7,15,.01,.9,0.,SEED)
bestW,bestb=best7
print("best epoch:",int(np.argmax(hist7["val_acc"]))+1)
print("best validation accuracy:",max(hist7["val_acc"]))
print("test accuracy:",cifar_eval(Xc_test,yc_test,bestW,bestb)[1])
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(hist7["train_loss"],label="train"); ax[0].plot(hist7["val_loss"],label="validation"); ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid()
ax[1].plot(hist7["train_acc"],label="train"); ax[1].plot(hist7["val_acc"],label="validation"); ax[1].set_title("Accuracy"); ax[1].legend(); ax[1].grid()
plt.tight_layout(); plt.show()
# First-layer neuron visualisations
fig,axs=plt.subplots(4,4,figsize=(8,8))
for i,ax in enumerate(axs.flat):
    im=bestW[0][:,i].reshape(3,32,32).transpose(1,2,0)
    im=(im-im.min())/(im.max()-im.min()+1e-12)
    ax.imshow(im); ax.axis("off")
plt.tight_layout(); plt.show()
pred=np.argmax(np.vstack([softmax_np(forward(Xc_test[i:i+256],bestW,bestb)[0][-1]) for i in range(0,len(yc_test),256)]),axis=1)
cm=np.zeros((10,10),dtype=int)
for a,b in zip(yc_test,pred): cm[a,b]+=1
class_acc=np.diag(cm)/np.maximum(cm.sum(axis=1),1)
print("confusion matrix:\n",cm); print("per-class accuracy:",class_acc)
print("best/worst class:",class_names[int(class_acc.argmax())],class_names[int(class_acc.argmin())])
# Regularised comparison
Wr,br=init_cifar(SEED); hr,_=train_cifar(Wr,br,15,.01,.9,1e-3,SEED)
print("train-validation gap lambda=0:",hist7["train_acc"][-1]-hist7["val_acc"][-1])
print("train-validation gap lambda=1e-3:",hr["train_acc"][-1]-hr["val_acc"][-1])

## P8. Gradient flow in deep PyTorch networks

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(SEED)
xb=torch.tensor(Xc_train[:256],dtype=torch.float32)
yb=torch.tensor(yc_train[:256],dtype=torch.long)
class DeepNet(nn.Module):
    def __init__(self,activation="relu",init="he",bn=False):
        super().__init__(); self.layers=nn.ModuleList(); self.norms=nn.ModuleList()
        self.activation=activation; self.bn=bn
        dims=[3072]+[128]*10+[10]
        for i in range(len(dims)-1):
            layer=nn.Linear(dims[i],dims[i+1]); self.layers.append(layer)
            if i<10 and bn: self.norms.append(nn.BatchNorm1d(128))
            if init=="he": nn.init.kaiming_normal_(layer.weight,nonlinearity="relu" if activation=="relu" else "linear")
            else: nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
    def forward(self,x):
        for i,layer in enumerate(self.layers):
            x=layer(x)
            if i<10:
                if self.bn: x=self.norms[i](x)
                x=torch.relu(x) if self.activation=="relu" else torch.sigmoid(x)
        return x
nets=[("sigmoid Xavier",DeepNet("sigmoid","xavier")),
      ("sigmoid He",DeepNet("sigmoid","he")),
      ("ReLU Xavier",DeepNet("relu","xavier")),
      ("ReLU He",DeepNet("relu","he"))]
grad_first={}
for name,net in nets:
    net.train(); net.zero_grad(set_to_none=True)
    loss=nn.CrossEntropyLoss()(net(xb),yb); loss.backward()
    vals=[layer.weight.grad.abs().mean().item() for layer in net.layers]
    grad_first[name]=vals[0]
    plt.plot(range(1,len(vals)+1),vals,marker="o",label=name)
plt.yscale("log"); plt.xlabel("Linear layer index"); plt.ylabel("Mean absolute weight gradient"); plt.legend(); plt.grid(); plt.show()
print("first-layer mean abs gradients:",grad_first)
# Forward hook standard deviations for ReLU-He and ReLU-Xavier
stds={}
for name,net in [("ReLU He",nets[3][1]),("ReLU Xavier",nets[2][1])]:
    vals=[]; hooks=[]
    for layer in net.layers:
        hooks.append(layer.register_forward_hook(lambda m,i,o,v=vals: v.append(o.detach().std().item())))
    with torch.no_grad(): net(xb)
    for h in hooks: h.remove()
    stds[name]=vals
    plt.plot(range(1,len(vals)+1),vals,marker="o",label=name)
plt.xlabel("Linear layer index"); plt.ylabel("Output standard deviation"); plt.legend(); plt.grid(); plt.show()
# BN sigmoid Xavier comparison
bnnet=DeepNet("sigmoid","xavier",bn=True)
bnnet.train(); bnnet.zero_grad(set_to_none=True)
nn.CrossEntropyLoss()(bnnet(xb),yb).backward()
print("sigmoid Xavier first-layer gradient without BN:",grad_first["sigmoid Xavier"])
print("sigmoid Xavier first-layer gradient with BN:",bnnet.layers[0].weight.grad.abs().mean().item())

## P9. Bonus — PyTorch autograd versus NumPy backpropagation

In [ ]:
# Same architecture, seed, first 8 standardised Wine samples, float64, lambda=.01.
npW,npb=init_mlp([13,5,4,3],SEED)
X9=X_train[:8].astype(np.float64); y9=y_train[:8]
l9,z9,a9=loss_np(X9,y9,npW,npb,lam=.01)
ngW,ngb,ngZ=backward(X9,y9,npW,npb,z9,a9,lam=.01)
tWs=[torch.tensor(w,dtype=torch.float64,requires_grad=True) for w in npW]
tbs=[torch.tensor(b,dtype=torch.float64,requires_grad=True) for b in npb]
tx=torch.tensor(X9,dtype=torch.float64); ty=torch.tensor(y9,dtype=torch.long)
acts=[]; cur=tx
for i,(w,b) in enumerate(zip(tWs,tbs)):
    cur=cur@w+b
    if i<len(tWs)-1:
        cur=torch.relu(cur); acts.append(cur)
    else: logits=cur
for a in acts: a.retain_grad()
loss_t=nn.functional.cross_entropy(logits,ty)+.01/2*sum((w*w).sum() for w in tWs)
loss_t.backward()
diffs=[]
for i in range(3):
    diffs += [np.max(np.abs(tWs[i].grad.numpy()-ngW[i])),
              np.max(np.abs(tbs[i].grad.numpy()-ngb[i]))]
print("PyTorch vs NumPy max parameter-gradient difference:",max(diffs))
assert max(diffs)<1e-9
# Retain Z gradients explicitly
cur=tx; zt=[]
for i,(w,b) in enumerate(zip(tWs,tbs)):
    zz=cur@w+b; zz.retain_grad(); zt.append(zz)
    cur=torch.relu(zz) if i<2 else zz
loss_z=nn.functional.cross_entropy(cur,ty)+.01/2*sum((w*w).sum() for w in tWs)
loss_z.backward()
print("max dZ differences:",[np.max(np.abs(zt[i].grad.numpy()-ngZ[i])) for i in range(3)])
# Graph traversal
seen=set(); stack=[loss_t.grad_fn]; nodes=[]
while stack:
    node=stack.pop()
    if node is None or node in seen: continue
    seen.add(node); nodes.append(type(node).__name__)
    stack.extend(n for n,_ in node.next_functions if n is not None)
print("backward graph node types:",nodes)
# Gradient accumulation
# Build a fresh graph, then backpropagate twice through that same graph.
for p in tWs+tbs: p.grad=None
logits_acc=tx
for i,(w,b) in enumerate(zip(tWs,tbs)):
    logits_acc=logits_acc@w+b
    if i<2: logits_acc=torch.relu(logits_acc)
loss_again=nn.functional.cross_entropy(logits_acc,ty)+.01/2*sum((w*w).sum() for w in tWs)
loss_again.backward(retain_graph=True)
g_once=tWs[0].grad.clone()
loss_again.backward()
print("gradient doubling max diff:",(tWs[0].grad-2*g_once).abs().max().item())
for p in tWs+tbs: p.grad=None
# no_grad and detach demonstrations
with torch.no_grad():
    no_grad_out=tx@tWs[0]+tbs[0]
print("no_grad output requires_grad:",no_grad_out.requires_grad)
first=(tx@tWs[0]+tbs[0]).relu().detach()
second=(first@tWs[1]+tbs[1]).relu()
third=second@tWs[2]+tbs[2]
det_loss=nn.functional.cross_entropy(third,ty)
for p in tWs+tbs: p.grad=None
det_loss.backward()
print("first-layer weight gradient after detaching first hidden activation:",
      None if tWs[0].grad is None else tWs[0].grad.abs().max().item())

## P10
The supplied PDF contains P1–P9 only. It mentions P4–P10 in the overview, but no P10 statement or tasks appear in the uploaded nine pages. No P10 work can be responsibly inferred. If a separate P10 page/file exists, attach it before submission.